In [0]:
BASE_DIR = '/Volumes/sales/landing/volumes/Product_Category_Translation/'
CHECKPOINT_DIR = '/Volumes/sales/landing/volumes/checkpoint_dir'
CHECKPOINT_LOCATION = f'{CHECKPOINT_DIR}/ProductCategoryTranslation'
CATALOG = 'SALES'
SCHEMA = 'BRONZE'
TARGET_TABLE = 'ProductCategoryTranslation'
print(f'Checkpoint Location: {CHECKPOINT_LOCATION}')
print(f'Base Location: {BASE_DIR}')

In [0]:
ProductCategoryTranslation_schema = """product_category_name STRING, 
                 product_category_name_english STRING
                 """

In [0]:
ProductCategoryTranslation_df = (spark.read.format('csv').option('header','true').schema(ProductCategoryTranslation_schema).load(f'{BASE_DIR}')
           )
display(ProductCategoryTranslation_df.limit(3))
ProductCategoryTranslation_df.printSchema()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

ProductCategoryTranslation_df = (
            spark.readStream.format('cloudFiles')
                 .option('cloudFiles.format', 'csv')
                 .option('cloudFiles.schemaEvolutionMode', 'rescue')
                 .option('header', 'true')
                 .schema(ProductCategoryTranslation_schema)
                 .load(f'{BASE_DIR}')
        )
ProductCategoryTranslation_trans_df = ProductCategoryTranslation_df.select(
    "product_category_name",
    "product_category_name_english",
    "_metadata.file_name",
    "_metadata.file_path",).withColumn('load_timestamp', current_timestamp())

In [0]:
qry = (
    ProductCategoryTranslation_trans_df.writeStream.format('delta')
    .outputMode('append')
    .option('checkpointLocation', CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(f'{CATALOG}.{SCHEMA}.{TARGET_TABLE}')
)
qry.awaitTermination()